# C3 — Colectare comentarii YouTube
În acest notebook colectăm un eșantion  de comentarii publice de pe YouTube.
Scopul nu este să obținem corpusul final mare, ci să înțelegem fluxul:
sursă → API → comentarii brute → fișier JSONL.
La final, fiecare student salvează propriul fișier în `data/raw/`.

## 1. Ce trebuie să avem pregătit
Avem nevoie de:
- fișier `.env` în root-ul proiectului
- cheia `YOUTUBE_API_KEY`
- un handle de canal YouTube
Exemplu în `.env`:
```text
YOUTUBE_API_KEY=cheia_ta_aici

In [24]:

from pathlib import Path
import os
import json
import requests
from datetime import datetime
from dotenv import load_dotenv

## 2. Încărcăm cheia API
Notebook-ul caută fișierul `.env` în root-ul proiectului.
Dacă cheia nu este găsită, colectarea nu poate porni.

In [25]:
ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")
API_KEY = os.getenv("YOUTUBE_API_KEY")
BASE_URL = "https://www.googleapis.com/youtube/v3"
print("Root proiect:", ROOT)
print("Cheie găsită:", API_KEY is not None)

Root proiect: /Users/catalinaminciuna/Library/CloudStorage/OneDrive-UniversitateaBabeş-Bolyai/masterat/inginerie AI/proiect AI Eng/echochamber-project-team3
Cheie găsită: True


## 3. Alegem canalul și numărul de videoclipuri
Fiecare student schimbă `student_id` și `handle`.
Pentru exercițiu folosim puține videoclipuri, ca să nu consumăm inutil cota API.

In [26]:
student_id = "student_02"
handle = "CălinGeorgescu-CanalulOficial"
max_videos = 10
max_comments_per_video = 10
output_file = ROOT / "data" / "raw" / f"{student_id}_youtube_raw.jsonl"
print(output_file)

/Users/catalinaminciuna/Library/CloudStorage/OneDrive-UniversitateaBabeş-Bolyai/masterat/inginerie AI/proiect AI Eng/echochamber-project-team3/data/raw/student_02_youtube_raw.jsonl


## 4. Găsim canalul YouTube

YouTube lucrează intern cu `channel_id`, nu direct cu numele canalului.
De aceea, primul pas este să transformăm handle-ul în `channel_id`.

In [27]:
channel_response = requests.get(
    f"{BASE_URL}/channels",
    params={
        "part": "id",
        "forHandle": handle,
        "key": API_KEY
    }
)
channel_data = channel_response.json()
channel_data

{'kind': 'youtube#channelListResponse',
 'etag': 'UFTKg9VFdUN4bLLNFOw49HFDgw0',
 'pageInfo': {'totalResults': 1, 'resultsPerPage': 5},
 'items': [{'kind': 'youtube#channel',
   'etag': '0L28yG9EWNGeZg-nQFCSk6LOYwc',
   'id': 'UCOJOwy9IEU6ELUJFHxVFLKQ'}]}

In [ ]:
channel_id = channel_data["items"][0]["id"]
channel_id

'UCOJOwy9IEU6ELUJFHxVFLKQ'

## 5. Luăm cele mai recente videoclipuri
Acum cerem ultimele videoclipuri publicate de canal.
Pentru curs folosim doar câteva videoclipuri.

In [29]:
videos_response = requests.get(
    f"{BASE_URL}/search",
    params={
        "part": "snippet",
        "channelId": channel_id,
        "type": "video",
        "order": "date",
        "maxResults": max_videos,
        "key": API_KEY
    }
)
videos_data = videos_response.json()
videos_data["items"][0]

{'kind': 'youtube#searchResult',
 'etag': 'NawIzVem3YRk7mne6BrM1URyGIw',
 'id': {'kind': 'youtube#video', 'videoId': 'r9LctgYNa2c'},
 'snippet': {'publishedAt': '2026-05-15T05:06:56Z',
  'channelId': 'UCOJOwy9IEU6ELUJFHxVFLKQ',
  'title': 'Călin Georgescu la Marius Tucă Show (14.05.2026 )',
  'description': 'MT: V-ați făcut dușmani? CG: Înseamnă că am învins. MT: Ați pierdut prieteni? CG: Înseamnă că nu am avut.',
  'thumbnails': {'default': {'url': 'https://i.ytimg.com/vi/r9LctgYNa2c/default.jpg',
    'width': 120,
    'height': 90},
   'medium': {'url': 'https://i.ytimg.com/vi/r9LctgYNa2c/mqdefault.jpg',
    'width': 320,
    'height': 180},
   'high': {'url': 'https://i.ytimg.com/vi/r9LctgYNa2c/hqdefault.jpg',
    'width': 480,
    'height': 360}},
  'channelTitle': 'Călin Georgescu - Canalul Oficial',
  'liveBroadcastContent': 'none',
  'publishTime': '2026-05-15T05:06:56Z'}}

In [ ]:
videos = []
for item in videos_data["items"]:
    videos.append({
        "video_id": item["id"]["videoId"],
        "video_title": item["snippet"]["title"],
        "video_date": item["snippet"]["publishedAt"][:10]
    })
videos

[{'video_id': 'r9LctgYNa2c',
  'video_title': 'Călin Georgescu la Marius Tucă Show (14.05.2026 )',
  'video_date': '2026-05-15'},
 {'video_id': 'e9dY7rhLG4c',
  'video_title': 'Călin Georgescu - România nu are un guvern pe măsura ei ( 28.04.2026 - la ICCJ )',
  'video_date': '2026-04-28'},
 {'video_id': 'ki0hPgSlbZ4',
  'video_title': 'Călin Georgescu și maestrul Ion Cristoiu la Realitatea TV ( 16.04.2026 )',
  'video_date': '2026-04-20'},
 {'video_id': 'VD1UoFcNcW0',
  'video_title': 'Călin Georgescu - Ni se interzice să le spunem hoților, hoți ( 2026.04.14 - la ICCJ )',
  'video_date': '2026-04-15'},
 {'video_id': 'HqOScz9dXP4',
  'video_title': 'Călin Georgescu - Sărbătoarea Învierii și unitatea românilor ✝ ( 12.04.2026 )',
  'video_date': '2026-04-11'},
 {'video_id': 'PM9d0b-yUVc',
  'video_title': 'Dragii mei, vă mulțumesc ! ( 09.04.2026 )',
  'video_date': '2026-04-09'},
 {'video_id': 'g9RhhLGdMq0',
  'video_title': 'Călin Georgescu și Anca Alexandrescu la Culisele Statului Paral

## 6. Colectăm comentariile
Pentru fiecare videoclip luăm comentariile publice ordonate după relevanță.
În acest exercițiu nu folosim paginare, deci luăm maximum 100 comentarii per videoclip.

In [31]:
comments = []
for video in videos:
    print("Colectez:", video["video_title"][:80])
    comments_response = requests.get(
        f"{BASE_URL}/commentThreads",
        params={
            "part": "snippet",
            "videoId": video["video_id"],
            "maxResults": max_comments_per_video,
            "textFormat": "plainText",
            "order": "relevance",
            "key": API_KEY
        }
    )
    comments_data = comments_response.json()
    for comment_item in comments_data.get("items", []):
        snippet = comment_item["snippet"]["topLevelComment"]["snippet"]
        record = {
            "id": f"yt_{video['video_id']}_{comment_item['id']}",
            "source_platform": "youtube",
            "source_channel": handle,
            "text_raw": snippet["textDisplay"],
            "video_id": video["video_id"],
            "video_title": video["video_title"],
            "video_date": video["video_date"],
            "comment_date": snippet["publishedAt"][:10],
            "likes": snippet["likeCount"],
            "collected_at": datetime.utcnow().strftime("%Y-%m-%d")
        }
        comments.append(record)
len(comments)

Colectez: Călin Georgescu la Marius Tucă Show (14.05.2026 )


/var/folders/rj/pj52rxcd7gddmhj55h5kcqnw0000gn/T/ipykernel_80713/3206550858.py:28: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "collected_at": datetime.utcnow().strftime("%Y-%m-%d")


Colectez: Călin Georgescu - România nu are un guvern pe măsura ei ( 28.04.2026 - la ICCJ )
Colectez: Călin Georgescu și maestrul Ion Cristoiu la Realitatea TV ( 16.04.2026 )
Colectez: Călin Georgescu - Ni se interzice să le spunem hoților, hoți ( 2026.04.14 - la I
Colectez: Călin Georgescu - Sărbătoarea Învierii și unitatea românilor ✝ ( 12.04.2026 )
Colectez: Dragii mei, vă mulțumesc ! ( 09.04.2026 )
Colectez: Călin Georgescu și Anca Alexandrescu la Culisele Statului Paralel ( 07.04.2026 )
Colectez: Temelia democrației este voința poporului
Colectez: Călin Georgescu - Adevărul nu se afirmă, se dovedește ( 21.01.2026 - la Curtea d
Colectez: Călin Georgescu - Dumnezeu veghează asupra poporului român ( 26.03.2026 - Judeca


100

## 6.1 Salvăm comentariile brute

In [32]:
raw_file = ROOT / "data" / "raw" / f"{student_id}_youtube_raw.jsonl"
raw_file.parent.mkdir(parents=True, exist_ok=True)

with raw_file.open("w", encoding="utf-8") as f:
    for c in comments:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")

print(f"✓ Am salvat {len(comments)} comentarii brute în:")
print(" ", raw_file)

✓ Am salvat 100 comentarii brute în:
  /Users/catalinaminciuna/Library/CloudStorage/OneDrive-UniversitateaBabeş-Bolyai/masterat/inginerie AI/proiect AI Eng/echochamber-project-team3/data/raw/student_02_youtube_raw.jsonl


# Explorare si curatare

## 7. Inspectăm primele comentarii
Înainte să salvăm fișierul, verificăm dacă datele arată cum trebuie.

In [33]:
comments[:3]

[{'id': 'yt_r9LctgYNa2c_UgybpC6ZPOGVmI2Q8vx4AaABAg',
  'source_platform': 'youtube',
  'source_channel': 'CălinGeorgescu-CanalulOficial',
  'text_raw': 'Daca systemului nu i-a placut ca CG sa castige alegerile cu 70%, va castiga cu 80%!\n💙💛❤️',
  'video_id': 'r9LctgYNa2c',
  'video_title': 'Călin Georgescu la Marius Tucă Show (14.05.2026 )',
  'video_date': '2026-05-15',
  'comment_date': '2026-05-15',
  'likes': 123,
  'collected_at': '2026-05-16'},
 {'id': 'yt_r9LctgYNa2c_UgzDT2V_izomGAvwVkR4AaABAg',
  'source_platform': 'youtube',
  'source_channel': 'CălinGeorgescu-CanalulOficial',
  'text_raw': 'DUMNEZEU SĂ VĂ BINECUVÂNTEZE!',
  'video_id': 'r9LctgYNa2c',
  'video_title': 'Călin Georgescu la Marius Tucă Show (14.05.2026 )',
  'video_date': '2026-05-15',
  'comment_date': '2026-05-15',
  'likes': 73,
  'collected_at': '2026-05-16'},
 {'id': 'yt_r9LctgYNa2c_UgzfWijCgHPP9eWotVR4AaABAg',
  'source_platform': 'youtube',
  'source_channel': 'CălinGeorgescu-CanalulOficial',
  'text_raw':

In [34]:
comments[0].keys()

dict_keys(['id', 'source_platform', 'source_channel', 'text_raw', 'video_id', 'video_title', 'video_date', 'comment_date', 'likes', 'collected_at'])

## 8. Curățare minimă a textului
Acum pornim de la `text_raw` și construim o variantă curățată în câmpul `text`.
Nu schimbăm sensul comentariului. Eliminăm doar zgomot simplu: linkuri, spații inutile, texte prea scurte și duplicate.

In [35]:
import re

def clean_text(text):
    text = re.sub(r"http\S+", "", text)      # elimină linkuri
    text = re.sub(r"\s+", " ", text)         # normalizează spațiile
    return text.strip()

## 9. Aplicăm curățarea
Pentru fiecare comentariu păstrăm textul original în `text_raw` și adăugăm textul curățat în `text`.

In [36]:
for comment in comments:
    comment["text"] = clean_text(comment["text_raw"])

comments[0]

{'id': 'yt_r9LctgYNa2c_UgybpC6ZPOGVmI2Q8vx4AaABAg',
 'source_platform': 'youtube',
 'source_channel': 'CălinGeorgescu-CanalulOficial',
 'text_raw': 'Daca systemului nu i-a placut ca CG sa castige alegerile cu 70%, va castiga cu 80%!\n💙💛❤️',
 'video_id': 'r9LctgYNa2c',
 'video_title': 'Călin Georgescu la Marius Tucă Show (14.05.2026 )',
 'video_date': '2026-05-15',
 'comment_date': '2026-05-15',
 'likes': 123,
 'collected_at': '2026-05-16',
 'text': 'Daca systemului nu i-a placut ca CG sa castige alegerile cu 70%, va castiga cu 80%! 💙💛❤️'}

## 10. Filtrăm comentariile prea scurte
Pentru exercițiu păstrăm doar comentariile care au cel puțin 60 de caractere.
Comentariile foarte scurte sunt greu de interpretat în analiza discursivă.

In [37]:
MIN_CHARS = 60

comments_clean = [
    comment for comment in comments
    if len(comment["text"]) >= MIN_CHARS
]

print("Comentarii brute:", len(comments))
print("Comentarii după filtrarea lungimii:", len(comments_clean))

Comentarii brute: 100
Comentarii după filtrarea lungimii: 51


## 11. Filtrăm textele cu prea puține litere
Comentariile formate mai ales din emoji, simboluri sau caractere izolate produc zgomot.
Păstrăm comentariile în care cel puțin 50% dintre caractere sunt litere.

In [39]:
MIN_ALPHA = 0.5

def alpha_ratio(text):
    if len(text) == 0:
        return 0
    letters = sum(char.isalpha() for char in text)
    return letters / len(text)

comments_clean = [
    comment for comment in comments_clean
    if alpha_ratio(comment["text"]) >= MIN_ALPHA
]

print("Comentarii după filtrarea literelor:", len(comments_clean))

Comentarii după filtrarea literelor: 51


## 12. Eliminăm duplicatele
Dacă același text apare de mai multe ori, îl păstrăm o singură dată.

In [40]:
seen_texts = set()
unique_comments = []

for comment in comments_clean:
    text = comment["text"].lower()
    if text not in seen_texts:
        unique_comments.append(comment)
        seen_texts.add(text)

comments_clean = unique_comments

print("Comentarii finale după deduplicare:", len(comments_clean))

Comentarii finale după deduplicare: 51


## 14. Salvăm fișierul curățat
Salvăm rezultatul în `data/cleaned/`.

In [41]:
clean_output_file = ROOT / "data" / "cleaned" / f"{student_id}_youtube_clean.jsonl"
clean_output_file.parent.mkdir(parents=True, exist_ok=True)

with clean_output_file.open("w", encoding="utf-8") as f:
    for comment in comments_clean:
        f.write(json.dumps(comment, ensure_ascii=False) + "\n")

print("Comentarii curate salvate:", len(comments_clean))
print("Fișier:", clean_output_file)

Comentarii curate salvate: 51
Fișier: /Users/catalinaminciuna/Library/CloudStorage/OneDrive-UniversitateaBabeş-Bolyai/masterat/inginerie AI/proiect AI Eng/echochamber-project-team3/data/cleaned/student_02_youtube_clean.jsonl


# Functia de curatare

In [42]:
import re

def clean_comments(comments, min_chars=60, min_alpha=0.5):
    cleaned = []
    seen_texts = set()
    
    for comment in comments:
        # 1. Curățare text
        text = comment["text_raw"]
        text = re.sub(r"http\S+", "", text)
        text = re.sub(r"\s+", " ", text).strip()
        
        # 2. Filtru lungime
        if len(text) < min_chars:
            continue
        
        # 3. Filtru proporție litere
        letters = sum(char.isalpha() for char in text)
        alpha_ratio = letters / len(text) if len(text) > 0 else 0
        
        if alpha_ratio < min_alpha:
            continue
        
        # 4. Filtru duplicate
        text_key = text.lower()
        if text_key in seen_texts:
            continue
        
        seen_texts.add(text_key)
        
        # 5. Păstrăm comentariul și adăugăm textul curățat
        new_comment = comment.copy()
        new_comment["text"] = text
        new_comment["lang"] = "ro"
        cleaned.append(new_comment)
    
    return cleaned

In [43]:
comments_clean = clean_comments(
    comments,
    min_chars=60,
    min_alpha=0.5
)

print("Comentarii brute:", len(comments))
print("Comentarii curate:", len(comments_clean))

Comentarii brute: 100
Comentarii curate: 51


In [44]:
for comment in comments_clean[:3]:
    print("RAW:", comment["text_raw"])
    print("CLEAN:", comment["text"])
    print("---")

RAW: Daca systemului nu i-a placut ca CG sa castige alegerile cu 70%, va castiga cu 80%!
💙💛❤️
CLEAN: Daca systemului nu i-a placut ca CG sa castige alegerile cu 70%, va castiga cu 80%! 💙💛❤️
---
RAW: Iubesc adevărul ❤,.. Călin Georgescu Președintele meu al României 🫡🇹🇩. Punct și atât.🫡🇹🇩
CLEAN: Iubesc adevărul ❤,.. Călin Georgescu Președintele meu al României 🫡🇹🇩. Punct și atât.🫡🇹🇩
---
RAW: Salutari Dl. Calin Georgescu.... Doamne Ajuta ! un om ales a lui Dumnezeu !
CLEAN: Salutari Dl. Calin Georgescu.... Doamne Ajuta ! un om ales a lui Dumnezeu !
---


In [45]:
clean_output_file = ROOT / "data" / "cleaned" / f"{student_id}_youtube_clean.jsonl"
clean_output_file.parent.mkdir(parents=True, exist_ok=True)

with clean_output_file.open("w", encoding="utf-8") as f:
    for comment in comments_clean:
        f.write(json.dumps(comment, ensure_ascii=False) + "\n")

print("Fișier salvat:", clean_output_file)
print("Comentarii salvate:", len(comments_clean))

Fișier salvat: /Users/catalinaminciuna/Library/CloudStorage/OneDrive-UniversitateaBabeş-Bolyai/masterat/inginerie AI/proiect AI Eng/echochamber-project-team3/data/cleaned/student_02_youtube_clean.jsonl
Comentarii salvate: 51


15. Ce am obținut
Am produs două fișiere:
- `data/raw/student_XX_youtube_raw.jsonl` — comentarii brute
- `data/cleaned/student_XX_youtube_clean.jsonl` — comentarii curățate
Fișierul curățat va putea fi unit cu fișierele celorlalți membri ai echipei.